# Lesson 27 — Machine Learning for Optimization

## Learning objectives

1. Distinguish *learning to branch*, *learning to cut*, *learning primal heuristics*.
2. Recognize the role of GNNs for instance representation {cite:p}`Gasse2019`.
3. Understand decision-focused vs predict-then-optimize learning {cite:p}`Elmachtoub2022,Wilder2019`.
4. Critically read a "ML beats Gurobi" paper.

## 1. The taxonomy

{cite:p}`Bengio2021` organize ML-for-optimization into:

- **End-to-end learning**: predict the solution from instance features. Limited to easy problems.
- **Configuring** combinatorial algorithms: choose hyperparameters per instance.
- **Interleaved with B&B**: learn branching, cuts, heuristics inside the tree.

## 2. Learning to branch

Replace strong branching (Lesson 14) with a fast classifier trained to imitate it. {cite:p}`Gasse2019` use a GNN over the variable-constraint bipartite graph.

**Training:** supervised — at each B&B node, log strong-branching scores, train classifier to predict argmax. **Inference:** use the classifier instead of strong branching, getting strong-branching quality at fraction of the cost.

Reported speedups: 2–5x on benchmark MILPs.

**A cautionary data point from `discopt` itself.** Through 0.6, `solve()`
accepted `branching_policy="gnn"`. No trained weights ever shipped, so the
"GNN" fell back to most-fractional branching and the option silently did
nothing. It was removed in 0.7: the developers' measurements showed that on
the instances where `discopt` is slow the bottleneck is bounding and range
reduction, not branching — even real strong branching did not help — so a
learned imitation of strong branching could not pay off there. Default
branching is reliability branching (pseudocosts, priorities, and strong
branching) in the Rust tree manager, and there is no user branching
callback, so a learned rule cannot be plugged into `discopt`'s tree.
What you *can* do on a pure MILP is measure the trade-off a learned rule
is meant to win: `m.solve(milp_backend="native", branching_rule=...)` runs
`discopt`'s own MILP tree with `"most_fractional"`, `"least_fractional"`,
`"pseudocost"` (the default) or `"strong"`, and reports strong branching's
probe LPs in `r.solver_stats["branching/strong_probe_lps"]` (they are not
counted in `node_count`). Ask "is branching the bottleneck *on my
instances*?" before learning a branching rule.

## 3. Learning primal heuristics

Train a model to *propose* good integer-feasible incumbents from the LP relaxation. Often used early in B&B. {cite:p}`Bengio2021` survey several approaches.

## 4. Decision-focused learning

When the *output* of a predictive model is fed into an optimizer, training the predictor to minimize prediction error is suboptimal — what matters is the *decision quality*. {cite:p}`Wilder2019,Mandi2020,Elmachtoub2022` develop *decision-focused* training: differentiate the optimizer (Lesson 26) and propagate gradients.

## 5. Reading the literature critically

ML-for-optimization papers vary widely in claim quality. Checklist:

- **Compared to what?** "Beats Gurobi default" without solver tuning is weak.
- **Generalization?** Trained and tested on the same instance distribution? Or transfers?
- **Compute cost?** Often the ML method's training cost is omitted.
- **Statistical reporting?** Confidence intervals on benchmarks are rare in ML papers.

For a sober view, see {cite:p}`Lu2024,Sandrin2025`. The field is rapidly evolving; today's "breakthrough" is often tomorrow's "moderate improvement".

## References
{cite:p}`Bengio2021,Gasse2019,Elmachtoub2022,Wilder2019,Mandi2020`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# A real knapsack solve to anchor the discussion.
import numpy as np, discopt as do

m = do.Model("knapsack20")
n = 20
rng = np.random.default_rng(0)
v = rng.integers(1, 30, n); w = rng.integers(1, 20, n); W = int(0.4 * w.sum())
x = m.binary("x", shape=(n,))
m.subject_to(w @ x <= W); m.maximize(v @ x)
r = m.solve()
print("status:", r.status, "  z*:", r.objective, "  nodes:", r.node_count)
print("route :", r.algorithm_route)   # pure MILPs go to HiGHS by default

# What a learned rule would imitate: strong branching on discopt's own MILP
# tree. A multidimensional knapsack (a single-row knapsack's LP has only one
# fractional variable, so every rule would pick the same one).
def mknap(seed=3, n=35, k=3):
    rng = np.random.default_rng(seed)
    A = rng.integers(5, 40, (k, n)); val = rng.integers(10, 50, n)
    mk = do.Model("mknap")
    y = mk.binary("y", shape=(n,))
    mk.subject_to(A @ y <= (0.5 * A.sum(axis=1)).astype(int))
    mk.maximize(val @ y)
    return mk
print(f"HiGHS (default)      nodes={mknap().solve().node_count}")
for rule in ["most_fractional", "pseudocost", "strong"]:
    rr = mknap().solve(milp_backend="native", branching_rule=rule, milp_cuts=False)
    probes = int(rr.solver_stats.get("branching/strong_probe_lps", 0))
    print(f"native {rule:15s} nodes={rr.node_count:4d}  probe LPs={probes:5d}  "
          f"time={rr.wall_time:.2f}s  z*={rr.objective:g}")
# Strong branching shrinks the tree but pays for it in probe LPs: the gap a
# cheap learned imitation of strong branching is trying to exploit.

# The pre-0.7 `branching_policy` option is gone, and solve() rejects unknown
# options instead of silently ignoring them:
try:
    m.solve(branching_policy="gnn")
except TypeError as e:
    print("TypeError:", str(e).split(".")[0])


Where ML *does* plug into `discopt` today:

- **ML inside the model** — trained networks and tree ensembles embedded as
  constraints with `discopt.ml` (Lesson 24), and `solver="surrogate"` for
  expensive black-box objectives.
- **Optimization inside the ML pipeline** — KKT sensitivities and JAX
  layers (`m.sensitivity(...).as_layer()`, Lesson 26), the ingredient
  decision-focused learning needs.
- **Learned relaxations: retired.** Up to 0.9, `solve(use_learned_relaxations=True)`
  swapped McCormick envelopes for ICNN-based learned convex relaxations.
  No pretrained models ever shipped, so the option never took effect, and
  discopt 0.10 removed it together with the relaxation route it served (it
  now raises `ValueError`). This is a useful data point for the writing
  assignment: a learned relaxation must be *provably* below the function
  everywhere on the box to give a valid bound, which is hard to guarantee
  for a trained network.
- **Observation hooks** — `node_callback` and `cut_callback` expose node
  relaxation solutions (`ctx.x_relaxation`, `ctx.node_bound`) and boxes,
  the raw material for node features or learned cuts. HiGHS, the default
  engine for a pure MILP, runs no callbacks, so passing one sends a MILP to
  `discopt`'s own search instead: `node_callback` to the MILP tree,
  `cut_callback` / `lazy_constraints` / `incumbent_callback` to spatial
  branch and bound (`r.algorithm_route` says which). The hooks observe and
  cut; they do not choose the branching variable, so per-node
  strong-branching labels still need your own tree, as in the exercises.